# N11 — Overtake Probability: EDA & Labeling

The goal of this notebook is to build the labeled dataset for the overtake probability
model (N12). We work at the level of **car pairs (X, Y)** per lap: X is the chasing
driver, Y is the car directly ahead. For each pair and each lap we determine whether
a real on-track overtake occurred.

The output is a Parquet file with contextual features and a binary `overtake` label,
ready for LightGBM training in N12.

**Data sources:**
- FastF1: positions, lap times, compound, pit stops, track status (2023–2025)
- OpenF1 `/v1/intervals`: real-time gap to the car ahead, aggregated per lap

**Exports:**
- EDA plots → `notebooks/strategy/overtake_probability/outputs/`
- Labeled dataset → `data/processed/overtake_labeled/`


---

## Step 0 — Setup

Standard imports, repo root resolution, and path definitions. FastF1 cache is pointed
at the existing `data/cache/fastf1` directory so we reuse already-downloaded sessions.


In [5]:
# ── Step 0 — Setup ────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import pathlib
import requests
import time
import json


import fastf1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# ── Repo root ─────────────────────────────────────────────────────────────────
repo_root = pathlib.Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUTS   = repo_root / "notebooks" / "strategy" / "overtake_probability" / "outputs"
PROCESSED = repo_root / "data" / "processed" / "overtake_labeled"
CACHE     = repo_root / "data" / "cache" / "fastf1"

OUTPUTS.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

# ── FastF1 cache ──────────────────────────────────────────────────────────────
fastf1.Cache.enable_cache(str(CACHE))

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
sns.set_palette("tab10")

print("repo_root :", repo_root)
print("OUTPUTS   :", OUTPUTS)
print("PROCESSED :", PROCESSED)
print("CACHE     :", CACHE)


repo_root : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager
OUTPUTS   : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\notebooks\strategy\overtake_probability\outputs
PROCESSED : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\processed\overtake_labeled
CACHE     : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\cache\fastf1


---

## Step 1 — Load FastF1 Race Data

We load all Race sessions for 2023, 2024, and 2025. For each session we extract the
laps DataFrame and tag it with year and GP name. The columns we care about are:
position, lap time, compound, tyre life, pit stop flags, track status, speed trap,
and the cumulative race timestamp (`Time`) which we'll use later to compute inter-car gaps.

Loading all sessions takes a few minutes on first run — FastF1 fetches from cache after that.


In [6]:
# ── Step 1 — Load FastF1 race laps (2023–2025) ───────────────────────────────
YEARS = [2023, 2024, 2025]
COLS  = [
    "DriverNumber", "Position", "LapNumber", "LapTime",
    "Time",           # cumulative race timestamp at lap end
    "Compound", "TyreLife",
    "PitInTime", "PitOutTime",
    "TrackStatus", "SpeedST", "IsAccurate",
]


def load_session_laps(year: int, gp: str, cols: list) -> pd.DataFrame | None:
    """Load Race laps for a single GP. Returns None on failure."""
    try:
        session = fastf1.get_session(year, gp, "R")
        session.load(laps=True, telemetry=False, weather=False, messages=False)
        laps = session.laps[cols].copy()
        laps["Year"]    = year
        laps["GP_Name"] = gp
        return laps
    except Exception as e:
        return None


def load_all_laps(years: list, cols: list) -> tuple[pd.DataFrame, list]:
    """
    Iterate over all GPs for each year and concatenate laps.
    Returns (df_raw, skipped) where skipped is a list of (year, gp) tuples.
    """
    records, skipped = [], []
    for year in years:
        schedule = fastf1.get_event_schedule(year, include_testing=False)
        for gp in schedule["EventName"].tolist():
            laps = load_session_laps(year, gp, cols)
            if laps is not None:
                records.append(laps)
            else:
                skipped.append((year, gp))
    return pd.concat(records, ignore_index=True), skipped


def clean_raw_laps(df: pd.DataFrame) -> pd.DataFrame:
    """Cast types and add LapTime_s column."""
    df = df.copy()
    df["LapTime_s"]    = df["LapTime"].dt.total_seconds()
    df["Position"]     = pd.to_numeric(df["Position"], errors="coerce")
    df["DriverNumber"] = df["DriverNumber"].astype(str)
    return df

In [7]:
# ── Run ───────────────────────────────────────────────────────────────────────
df_raw, skipped = load_all_laps(YEARS, COLS)
df_raw = clean_raw_laps(df_raw)

print(f"Loaded : {df_raw.shape[0]:,} laps  |  {df_raw['GP_Name'].nunique()} GPs  |  years {YEARS}")
print(f"Skipped: {len(skipped)}", skipped if skipped else "")
print()
print(df_raw[["Year","GP_Name","DriverNumber","LapNumber",
              "Position","LapTime_s","Compound","TyreLife"]].head(10).to_string(index=False))


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data

Loaded : 77,720 laps  |  24 GPs  |  years [2023, 2024, 2025]
Skipped: 0 

 Year            GP_Name DriverNumber  LapNumber  Position  LapTime_s Compound  TyreLife
 2023 Bahrain Grand Prix            1        1.0       1.0     99.019     SOFT       4.0
 2023 Bahrain Grand Prix            1        2.0       1.0     97.974     SOFT       5.0
 2023 Bahrain Grand Prix            1        3.0       1.0     98.006     SOFT       6.0
 2023 Bahrain Grand Prix            1        4.0       1.0     97.976     SOFT       7.0
 2023 Bahrain Grand Prix            1        5.0       1.0     98.035     SOFT       8.0
 2023 Bahrain Grand Prix            1        6.0       1.0     97.986     SOFT       9.0
 2023 Bahrain Grand Prix            1        7.0       1.0     98.021     SOFT      10.0
 2023 Bahrain Grand Prix            1        8.0       1.0     98.154     SOFT      11.0
 2023 Bahrain Grand Prix            1        9.0       1.0     98.278     SOFT      12.0
 2023 Bahrain Grand Prix            

---

## Step 2 — Building Car Pairs (X, Y)

For each lap and each driver X, we identify the car directly ahead (Y = driver with
`Position = Position_X - 1`) and compute the contextual features of that battle.

The gap between X and Y is derived from FastF1's `Time` column, which records the
cumulative race timestamp at the moment each driver crosses the finish line at the
end of each lap. The difference `Time_X - Time_Y` gives the true on-track gap in seconds.

We also map FastF1's relative compound names (SOFT/MEDIUM/HARD) to the circuit-specific
absolute compound IDs (C1–C6) using the mapping built during tire degradation work.
Pairs where the gap exceeds 5 seconds are discarded — beyond that threshold the two
drivers are not in an active battle and an overtake is physically irrelevant.


#### Load and Mapping

In [8]:
# ── Step 2 — Build car pairs (X, Y) ──────────────────────────────────────────

# ── Load compound mapping from canonical JSON ─────────────────────────────────
with open(repo_root / "data" / "tire_compounds_by_race.json", encoding="utf-8") as f:
    _tire_json = json.load(f)
TIRE_COMPOUNDS = {k: v for k, v in _tire_json.items() if not k.startswith("_")}

# ── Load circuit cluster mapping ──────────────────────────────────────────────
_df_clusters = pd.read_parquet(
    repo_root / "data" / "processed" / "circuit_clustering" / "circuit_clusters_k4.parquet"
)
CLUSTER_LOOKUP = dict(zip(_df_clusters["GP_Name"], _df_clusters["Cluster"].astype(int)))

# ── Bridge: FastF1 full event name → canonical short name ────────────────────
FULL_TO_SHORT = {
    "Australian Grand Prix":       "Melbourne",
    "Bahrain Grand Prix":          "Sakhir",
    "Saudi Arabian Grand Prix":    "Jeddah",
    "Azerbaijan Grand Prix":       "Baku",
    "Miami Grand Prix":            "Miami",
    "Emilia Romagna Grand Prix":   "Imola",
    "Monaco Grand Prix":           "Monaco",
    "Spanish Grand Prix":          "Barcelona",
    "Canadian Grand Prix":         "Montréal",
    "Austrian Grand Prix":         "Spielberg",
    "British Grand Prix":          "Silverstone",
    "Hungarian Grand Prix":        "Budapest",
    "Belgian Grand Prix":          "Spa-Francorchamps",
    "Dutch Grand Prix":            "Zandvoort",
    "Italian Grand Prix":          "Monza",
    "Singapore Grand Prix":        "Marina Bay",
    "Japanese Grand Prix":         "Suzuka",
    "Qatar Grand Prix":            "Lusail",
    "United States Grand Prix":    "Austin",
    "Mexico City Grand Prix":      "Mexico City",
    "São Paulo Grand Prix":        "São Paulo",
    "Las Vegas Grand Prix":        "Las Vegas",
    "Abu Dhabi Grand Prix":        "Yas Island",
    "Chinese Grand Prix":          "Shanghai",
}

#### Helper Functions Definition

In [9]:
# ── Helper functions ──────────────────────────────────────────────────────────
def get_absolute_compound(gp_full: str, year: int, compound: str) -> str:
    short = FULL_TO_SHORT.get(gp_full)
    if short is None:
        return "UNKNOWN"
    return TIRE_COMPOUNDS.get(str(year), {}).get(short, {}).get(compound, "UNKNOWN")


def get_cluster(gp_full: str) -> int:
    short = FULL_TO_SHORT.get(gp_full)
    return CLUSTER_LOOKUP.get(short, -1)


def build_pairs_for_race(race_df: pd.DataFrame) -> pd.DataFrame:
    """Build one (X, Y) pair row per active battle per lap for a single race."""
    race_df  = race_df.copy()
    race_df["Time_s"] = race_df["Time"].dt.total_seconds()
    gp_full  = race_df["GP_Name"].iloc[0]
    year     = int(race_df["Year"].iloc[0])
    cluster  = get_cluster(gp_full)

    pairs = []
    for lap_num, lap_group in race_df.groupby("LapNumber"):
        lap_group = lap_group.dropna(subset=["Position", "LapTime_s", "Time_s"])
        lap_group = lap_group[lap_group["Position"] > 0]
        pos_to_row = {row["Position"]: row for _, row in lap_group.iterrows()}

        for _, row_x in lap_group.iterrows():
            row_y = pos_to_row.get(row_x["Position"] - 1)
            if row_y is None:
                continue                           # X is leader or Y missing
            gap = abs(row_x["Time_s"] - row_y["Time_s"])
            if gap > 5.0:
                continue                           # not an active battle

            pairs.append({
                "Year":             year,
                "GP_Name":          gp_full,
                "LapNumber":        int(lap_num),
                "driver_x":         row_x["DriverNumber"],
                "driver_y":         row_y["DriverNumber"],
                "position_x":       row_x["Position"],
                "gap_ahead_s":      round(gap, 3),
                "pace_delta_s":     round(row_x["LapTime_s"] - row_y["LapTime_s"], 3),
                "tyre_life_x":      row_x["TyreLife"],
                "tyre_life_y":      row_y["TyreLife"],
                "tyre_life_diff":   row_x["TyreLife"] - row_y["TyreLife"],
                "compound_x":       get_absolute_compound(gp_full, year, row_x["Compound"]),
                "compound_y":       get_absolute_compound(gp_full, year, row_y["Compound"]),
                "speed_trap_delta": row_x["SpeedST"] - row_y["SpeedST"],
                "track_status":     row_x["TrackStatus"],
                "pit_in_x":         pd.notna(row_x["PitInTime"]),
                "pit_out_x":        pd.notna(row_x["PitOutTime"]),
                "pit_in_y":         pd.notna(row_y["PitInTime"]),
                "pit_out_y":        pd.notna(row_y["PitOutTime"]),
                "circuit_cluster":  cluster,
            })

    return pd.DataFrame(pairs)


def build_all_pairs(df: pd.DataFrame) -> pd.DataFrame:
    """Iterate over all (Year, GP_Name) race groups and concatenate pair rows."""
    groups    = df.groupby(["Year", "GP_Name"])
    all_pairs = []
    print(f"Building pairs for {len(groups)} races...")
    for (year, gp), race_df in groups:
        pairs = build_pairs_for_race(race_df)
        if not pairs.empty:
            all_pairs.append(pairs)
    return pd.concat(all_pairs, ignore_index=True)

In [10]:
# ── Run ───────────────────────────────────────────────────────────────────────
df_pairs_raw = build_all_pairs(df_raw)

print(f"\nPairs built  : {len(df_pairs_raw):,}")
print(f"Races        : {df_pairs_raw.groupby(['Year','GP_Name']).ngroups}")
print(f"UNKNOWN cmpd : {(df_pairs_raw['compound_x'] == 'UNKNOWN').sum()}")
print(f"Unknown clust: {(df_pairs_raw['circuit_cluster'] == -1).sum()}")
print()
print(df_pairs_raw[["Year","GP_Name","LapNumber","driver_x","driver_y",
                     "gap_ahead_s","pace_delta_s","compound_x","circuit_cluster"]
                   ].head(8).to_string(index=False))


Building pairs for 70 races...

Pairs built  : 54,283
Races        : 70
UNKNOWN cmpd : 3494
Unknown clust: 0

 Year              GP_Name  LapNumber driver_x driver_y  gap_ahead_s  pace_delta_s compound_x  circuit_cluster
 2023 Abu Dhabi Grand Prix          1       16        1        0.929         0.929         C4                1
 2023 Abu Dhabi Grand Prix          1       63        4        0.467         0.467         C4                1
 2023 Abu Dhabi Grand Prix          1       11       44        0.518         0.518         C4                1
 2023 Abu Dhabi Grand Prix          1        4       81        0.427         0.427         C4                1
 2023 Abu Dhabi Grand Prix          1       81       16        0.763         0.763         C4                1
 2023 Abu Dhabi Grand Prix          1       14       22        0.609         0.609         C4                1
 2023 Abu Dhabi Grand Prix          1       22       63        0.659         0.659         C4                1
 2

In [11]:
# Pega en una celda temporal para diagnóstico
unknown_mask = df_pairs_raw["compound_x"] == "UNKNOWN"
print("UNKNOWN rows:", unknown_mask.sum())
print("\nCompounds originales en filas UNKNOWN:")
# necesitamos volver a df_raw para ver el compound original
# check qué GPs 2025 tienen UNKNOWN
print(df_pairs_raw[unknown_mask].groupby(["Year","GP_Name"]).size().sort_values(ascending=False).head(15))


UNKNOWN rows: 3494

Compounds originales en filas UNKNOWN:
Year  GP_Name              
2024  São Paulo Grand Prix     838
      Canadian Grand Prix      657
2025  Australian Grand Prix    509
      British Grand Prix       420
      Miami Grand Prix         378
      Belgian Grand Prix       217
2023  Dutch Grand Prix         213
      Monaco Grand Prix        145
2024  British Grand Prix        95
2023  Canadian Grand Prix       22
dtype: int64


### Step 2 — Observations

**Pair construction** produced 54,283 active battle observations across 70 races
(2023–2025), covering all laps where the gap between consecutive positions was ≤ 5 s.
Zero unknown clusters confirm the circuit lookup is complete.

**UNKNOWN compounds (3,494 rows, ~6.4%)** are concentrated in known wet-weather races:
São Paulo 2024, Canada 2024, Australia 2025, Britain 2025, Monaco 2023, Spa 2025, etc.
FastF1 reports `INTERMEDIATE` or `WET` for these laps, which fall outside the
SOFT/MEDIUM/HARD mapping in `tire_compounds_by_race.json`. This is expected — these
compounds are not modelled and wet-race overtaking dynamics are fundamentally different
(reduced visibility, aquaplaning, train effects). These rows will be dropped in Step 3.

**Lap 1 artefact:** on the opening lap, `gap_ahead_s == pace_delta_s` by construction —
since `Time_s` equals `LapTime_s` when no previous laps have accumulated. These laps
are discarded by the `LapNumber > 3` filter in the labeling pipeline.

After Step 3 filtering the working dataset will be approximately **50,700 clean pairs**.
